# 08 — Direct Preference Optimization

## Objective
Verify the preference-loss gradient using small tensors, load real Med-REFL-DPO preference pairs, and optionally fine-tune the same Supervised Fine-Tuning (SFT) checkpoint used by the Group Relative Policy Optimization (GRPO) branch.

## Data and methodological boundaries
Med-REFL-DPO contains automatically constructed preference pairs; it is not a dataset of independently verified human preference annotations. The model learns relative preferences that may differ from the PubMedQA yes/no/maybe decision task. Measured improvements must therefore be established on an independent benchmark, not inferred from the loss alone.

The DPO (Direct Preference Optimization) objective penalizes policy log-probabilities that favor rejected responses relative to preferred ones, compared with a reference policy:

$$
\mathcal{L}_{\mathrm{DPO}} = -\mathbb{E}_{(x,y_w,y_l)}\left[\log\sigma\left(\beta\left(\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)}-\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}\right)\right)\right].
$$

Here, $x$ is the prompt, $y_w$ is the preferred completion, $y_l$ the rejected completion, $\pi_\theta$ the optimized policy, $\pi_{\mathrm{ref}}$ a reference policy, and $\beta$ a positive objective scale.

### Confirm the gradient direction on a toy preference pair

In [1]:
import torch

beta = 0.1
logp_chosen = torch.tensor(-2.0, requires_grad=True)
logp_rejected = torch.tensor(-2.5, requires_grad=True)
logp_ref_chosen = -2.2
logp_ref_rejected = -2.3

preference_margin = beta * (
    (logp_chosen - logp_ref_chosen)
    - (logp_rejected - logp_ref_rejected)
)
loss = -torch.nn.functional.logsigmoid(preference_margin)
loss.backward()

print("Illustrative loss:", round(float(loss.item()), 5))
print("Preferred gradient:", float(logp_chosen.grad.item()))
print("Rejected gradient:", float(logp_rejected.grad.item()))
assert logp_chosen.grad.item() < 0
assert logp_rejected.grad.item() > 0
# Lower loss favors increasing the preferred log-probability relative to rejected.

Illustrative loss: 0.67335
Preferred gradient: -0.04900013282895088
Rejected gradient: 0.04900013282895088


### Experiment locations and source-data contract

In [2]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


In [3]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter

DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

### Train on verified preference pairs

The local TRL DPO trainer cannot import with this PyTorch build (`DTensor` is missing), and the installed Transformers four-bit path is incompatible with this PyTorch build. This notebook therefore implements the same pairwise DPO loss directly with a bfloat16 base model and trainable PEFT LoRA adapter. The frozen reference is the saved SFT checkpoint: reference log-probabilities are precomputed before any DPO update. Each sampled pair has a distinct exact prompt; both full responses must fit the 1024-token bound. This short run is a method smoke test, not an independent preference or biomedical evaluation.

In [4]:
import os
import json
import time
import random
import hashlib
from collections import defaultdict

if os.getenv("RUN_TRAINING", "0") == "1":
    import numpy as np
    import torch.nn.functional as F
    from datasets import Dataset, load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    import transformers, peft, datasets

    started = time.perf_counter()
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    sft_dir = RUNS / "sft_adapter"
    lineage = json.loads((sft_dir / "run_manifest.json").read_text(encoding="utf-8"))
    preference_manifest = json.loads((RESEARCH / "data" / "med_refl_reasoning_manifest.json").read_text(encoding="utf-8"))
    if lineage["model_id"] != model_id or lineage["dataset_fingerprint"] != json.loads(MANIFEST.read_text(encoding="utf-8"))["fingerprint"]:
        raise ValueError("SFT model or PubMedQA source differs from the frozen training lineage")
    revision = lineage["model_revision"]
    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        cache = Path.home() / ".cache/huggingface/datasets/HANI-LAB___med-refl-dpo/reasoning_enhancement/0.0.0/001e469d8849a7c38a13797b9a52d550ffc36124/med-refl-dpo-train.arrow"
        if hashlib.sha256(cache.read_bytes()).hexdigest() != preference_manifest["cache_arrow_sha256"]:
            raise ValueError("Preference Arrow cache differs from notebook 07")
        source = Dataset.from_file(str(cache))
        source_location = str(cache)
    else:
        source = load_dataset("HANI-LAB/Med-REFL-DPO", "reasoning_enhancement", split="train")
        source_location = "Hugging Face Datasets load_dataset"
    if len(source) != preference_manifest["source_rows"] or not {"instruction", "input", "chosen", "rejected"}.issubset(source.column_names):
        raise ValueError("Preference source size/schema differs from notebook 07")
    tokenizer = AutoTokenizer.from_pretrained(str(sft_dir), local_files_only=True)
    groups = defaultdict(list)
    for i, row in enumerate(source):
        prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
        if not prompt or not row["chosen"] or not row["rejected"] or row["chosen"] == row["rejected"]:
            raise ValueError(f"Invalid preference row {i}")
        groups[prompt].append(i)
    if len(groups) != preference_manifest["unique_prompts"]:
        raise ValueError("Preference prompt groups differ from notebook 07")
    max_pairs = int(os.getenv("DPO_TRAIN_MAX_ITEMS", "120"))
    max_steps = int(os.getenv("DPO_MAX_STEPS", "30"))
    accumulation = 4
    max_length = 1024
    if not 1 <= max_pairs <= len(groups) or not 1 <= max_steps or max_steps * accumulation > max_pairs:
        raise ValueError("Require positive steps and at least four distinct prompt groups per step")
    group_keys = list(groups)
    np.random.default_rng(SEED).shuffle(group_keys)
    records = []
    screened = 0
    for prompt in group_keys:
        indices = list(groups[prompt])
        np.random.default_rng(SEED + len(prompt)).shuffle(indices)
        prefix = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True).input_ids
        for index in indices:
            screened += 1
            row = source[index]
            completions = []
            for field in ("chosen", "rejected"):
                ids = prefix + tokenizer(str(row[field]).strip(), add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
                completions.append(ids)
            if max(map(len, completions)) <= max_length:
                records.append({"source_row": index, "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                                "prefix_tokens": len(prefix), "chosen_ids": completions[0], "rejected_ids": completions[1]})
                break
            if screened >= 100000:
                raise RuntimeError("Length screening exceeded safety bound")
        if len(records) >= max_pairs:
            break
    if len(records) < max_pairs:
        raise ValueError("Too few distinct groups with two complete responses under token bound")
    assert len({r["prompt_sha256"] for r in records}) == len(records)
    print("Distinct prompt groups:", len(groups), "selected pairs:", len(records), "screened rows:", screened)
    print("Selected maximum sequence tokens:", max(max(len(r["chosen_ids"]), len(r["rejected_ids"])) for r in records))

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("This bounded training run requires a CUDA GPU with bfloat16 support")
    torch.cuda.reset_peak_memory_stats()
    base = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, local_files_only=True,
                                                torch_dtype=torch.bfloat16).to("cuda")
    policy = PeftModel.from_pretrained(base, str(sft_dir), is_trainable=True).to("cuda")
    policy.config.use_cache = False
    def completion_logp(ids, prefix_tokens):
        tokens = torch.tensor([ids], device="cuda")
        labels = tokens[:, 1:]
        logits = policy(input_ids=tokens, use_cache=False).logits[:, :-1, :].float()
        logp = F.log_softmax(logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
        return logp[:, prefix_tokens - 1:].sum()

    policy.eval()
    with torch.inference_mode():
        for record in records:
            record["ref_chosen"] = float(completion_logp(record["chosen_ids"], record["prefix_tokens"]).item())
            record["ref_rejected"] = float(completion_logp(record["rejected_ids"], record["prefix_tokens"]).item())
    policy.train()
    optimizer = torch.optim.AdamW((p for p in policy.parameters() if p.requires_grad), lr=5e-5)
    beta = 0.1
    logs = []
    for step in range(max_steps):
        optimizer.zero_grad(set_to_none=True)
        losses, margins = [], []
        for record in records[step * accumulation:(step + 1) * accumulation]:
            chosen = completion_logp(record["chosen_ids"], record["prefix_tokens"])
            rejected = completion_logp(record["rejected_ids"], record["prefix_tokens"])
            margin = beta * ((chosen - rejected) - (record["ref_chosen"] - record["ref_rejected"]))
            loss = -F.logsigmoid(margin)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite DPO loss")
            (loss / accumulation).backward()
            losses.append(float(loss.detach().item()))
            margins.append(float(margin.detach().item()))
        torch.nn.utils.clip_grad_norm_((p for p in policy.parameters() if p.requires_grad), 1.0)
        optimizer.step()
        log = {"step": step + 1, "loss": float(np.mean(losses)), "reward_margin": float(np.mean(margins)),
               "preference_accuracy": float(np.mean(np.array(margins) > 0))}
        logs.append(log)
        if (step + 1) % 5 == 0:
            print(log, flush=True)
    output_dir = RUNS / "dpo_adapter"
    output_dir.mkdir(exist_ok=True)
    policy.save_pretrained(str(output_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(output_dir))
    details = {"variant": "dpo", "method": "manual pairwise DPO, unquantized base with LoRA",
               "model_id": model_id, "model_revision": revision, "sft_checkpoint": str(sft_dir),
               "dataset_fingerprint": lineage["dataset_fingerprint"],
               "preference_dataset": "HANI-LAB/Med-REFL-DPO:reasoning_enhancement",
               "preference_fingerprint": preference_manifest["source_content_fingerprint"],
               "preference_cache_sha256": preference_manifest["cache_arrow_sha256"],
               "source_location": source_location, "seed": SEED,
               "selection": "seeded prompt-group shuffle; first complete fitting pair per group",
               "selected_source_rows": [r["source_row"] for r in records],
               "selected_prompt_sha256": [r["prompt_sha256"] for r in records],
               "n_preference_pairs": len(records), "screened_rows": screened,
               "max_length": max_length, "prompt_format": "Qwen user chat template plus assistant prefix; response plus EOS",
               "beta": beta, "learning_rate": 5e-5, "gradient_accumulation_steps": accumulation,
               "completed_steps": len(logs), "trainable_parameters": sum(p.numel() for p in policy.parameters() if p.requires_grad),
               "runtime_seconds": time.perf_counter() - started,
               "peak_gpu_gib": torch.cuda.max_memory_allocated() / 2**30,
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                            "peft": peft.__version__, "datasets": datasets.__version__},
               "run_status": "completed"}
    (output_dir / "run_manifest.json").write_text(json.dumps(details, indent=2) + "\n", encoding="utf-8")
    (output_dir / "log_history.json").write_text(json.dumps(logs, indent=2) + "\n", encoding="utf-8")
    print("DPO checkpoint saved:", output_dir)
else:
    print("DPO training disabled. Set RUN_TRAINING=1 for the bounded CUDA run.")

`torch_dtype` is deprecated! Use `dtype` instead!


Distinct prompt groups: 355 selected pairs: 120 screened rows: 522
Selected maximum sequence tokens: 1024


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'step': 5, 'loss': 0.6845482140779495, 'reward_margin': 0.04575653187930584, 'preference_accuracy': 0.75}
{'step': 10, 'loss': 0.7785796672105789, 'reward_margin': -0.16037445422261953, 'preference_accuracy': 0.25}
{'step': 15, 'loss': 0.7400734126567841, 'reward_margin': -0.035760494880378246, 'preference_accuracy': 0.25}
{'step': 20, 'loss': 0.5427026674151421, 'reward_margin': 0.3500503543764353, 'preference_accuracy': 0.75}
{'step': 25, 'loss': 0.5890211686491966, 'reward_margin': 0.2796081490814686, 'preference_accuracy': 0.75}
{'step': 30, 'loss': 0.7050520330667496, 'reward_margin': -0.007057191804051399, 'preference_accuracy': 0.5}
DPO checkpoint saved: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\dpo_adapter


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in Qwen/Qwen2.5-0.5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


### Results

The explicit DPO run selected 120 length-fitting, distinct-prompt Med-REFL-DPO pairs and completed 30 optimizer steps from SFT. Mean training loss was 0.6811 and in-batch preference accuracy 0.575. The saved adapter later scored 77/121 on PubMedQA validation.

### Interpretation

The run optimized the selected synthetic rankings and produced a compatible adapter. The PubMedQA score is a separate task outcome.

### Limitations

No independent Med-REFL preference-generalization result was established. Different training data and only four PubMedQA cases gained over SFT prevent attributing a robust improvement to DPO alone.

### Key Observations

Use notebook 12's matched comparison, not in-batch ranking, to describe answer-task behavior.


## Key observations

The saved adapter and training loss establish optimization on the selected synthetic preference pairs. They do not establish preference generalization or PubMedQA accuracy. Notebook 12 must evaluate the starting, SFT, and DPO models under the same frozen validation protocol. The selected examples have one exact prompt per group; semantically related or paraphrased cases could still overlap. The source's upstream revision and preference correctness remain unverified.